In [0]:
# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email = dbutils.secrets.get(scope="lit-scope-2", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="lit-scope-2", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="lit-scope-2", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="lit-scope-2", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
wildfire_file_path = "gs://lit-denser-data/raw/fires.tsv"
population_file_path = "gs://lit-denser-data/raw/usa_pd_2020_1km_UNadj_ASCII_XYZ.csv"

In [0]:
population_raw = sc.textFile(population_file_path)
fire_raw = sc.textFile(wildfire_file_path).map(lambda x: x.split("\t"))

In [0]:
population_raw.take(5)

In [0]:
fire_raw.take(3)

In [0]:
header = fire_raw.first()
index = {name: i for i, name in enumerate(header)}
index

In [0]:
COLUMNS = ["FOD_ID", "LATITUDE", "LONGITUDE", "STATE", "FIRE_SIZE", "FIRE_SIZE_CLASS", "NWCG_GENERAL_CAUSE", "DISCOVERY_DOY"]
kept_columns = [index[c] for c in COLUMNS]
row_length = len(header)
kept_columns

In [0]:
STATES = {"AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN",
             "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH",
             "NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT",
             "VT","VA","WA","WV","WI","WY"}

def clean_fire_row(row):
    if len(row) != row_length:
        return []
    fire_id, lat, lon, state, size, size_class, cause, discovery_day = (row[i] for i in kept_columns)
    try:
        lat, lon, size = float(lat), float(lon), float(size)
        discovery_day = int(float(discovery_day))
    except ValueError:
        return []
    state = state.strip().upper()
    if state not in STATES:
        return []
    return [(fire_id, lat, lon, state, size, size_class, cause, discovery_day)]

In [0]:
cleaned_fires = fire_raw.filter(lambda x: x[0] != "OBJECTID").flatMap(clean_fire_row)
cleaned_fires.take(2)

In [0]:
def clean_pop_row(row):
    parts = row.split(",")
    if len(parts) != 3:
        return []
    try:
        x, y, z = (float(p) for p in parts)
    except ValueError:
        return []                      # also drops the 'X,Y,Z' header
    return [(x, y, z)]

pop_clean = population_raw.flatMap(clean_pop_row)
pop_clean.take(2)

In [0]:
cleaned_fires.map(lambda x: "\t".join(map(str, x))) \
    .saveAsTextFile("gs://lit-denser-data/cleaned/fires")

pop_clean.map(lambda x: ",".join(map(str, x))) \
    .saveAsTextFile("gs://lit-denser-data/cleaned/population")